In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [2]:
NB_DIR = Path.cwd()
PROJ_ROOT = NB_DIR.parent
output_dir = PROJ_ROOT / 'synthetic_workloads'
os.makedirs(output_dir, exist_ok=True)

df = pd.read_parquet('../dataset/df_final.parquet')

In [3]:
MEAN_INTER_ARRIVAL_SECONDS = 15
MAX_WALLTIME_SECONDS = 72 * 3600

NUM_JOBS_PER_SIM = 10000
NUM_SIMULATIONS = 50

In [4]:
for i in range(NUM_SIMULATIONS):
    synthetic = df.sample(n=NUM_JOBS_PER_SIM, replace=True).copy()
    synthetic = synthetic.sample(frac=1).reset_index(drop=True)
    
    gaps = np.random.exponential(scale=MEAN_INTER_ARRIVAL_SECONDS, size=NUM_JOBS_PER_SIM)
    synthetic['submit_time'] = np.cumsum(gaps)
    
    overestimation_factor = np.random.uniform(1.2, 5.0, size=NUM_JOBS_PER_SIM)
    
    synthetic['requested_time'] = np.clip(synthetic['requested_time'], a_min=None, a_max=MAX_WALLTIME_SECONDS)
    
    synthetic['actual_runtime'] = np.maximum(1.0, synthetic['requested_time'] / overestimation_factor)
    
    cols_to_keep = [
        'submit_time', 'requested_processors', 'requested_time', 
        'actual_runtime', 'queue_number', 'think_time'
    ]
    final_workload = synthetic[cols_to_keep]
    
    file_path = os.path.join(output_dir, f'workload_{i+1:02d}.csv')
    final_workload.to_csv(file_path, index=False)

In [5]:
tdf = pd.read_csv('../synthetic_workloads/workload_01.csv')

In [6]:
tdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   submit_time           10000 non-null  float64
 1   requested_processors  10000 non-null  int64  
 2   requested_time        10000 non-null  int64  
 3   actual_runtime        10000 non-null  float64
 4   queue_number          10000 non-null  int64  
 5   think_time            10000 non-null  float64
dtypes: float64(3), int64(3)
memory usage: 468.9 KB


In [7]:
tdf.describe()

,submit_time,requested_processors,requested_time,actual_runtime,queue_number,think_time
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,1.000000e+04
mean,74842.836011,4.336900,35644.641700,13501.731143,2.749000,3.819231e+03
std,43616.600755,15.909749,64177.999325,27153.331917,1.314295,1.866775e+05
min,6.320144,1.000000,300.000000,63.849779,1.000000,0.000000e+00
25%,37150.979419,1.000000,7140.000000,1839.819635,2.000000,0.000000e+00
50%,74278.422747,1.000000,9000.000000,3420.492937,2.000000,0.000000e+00
75%,112823.314416,1.000000,16200.000000,7725.542979,3.000000,0.000000e+00
max,150151.413281,256.000000,259200.000000,213932.837312,6.000000,1.764246e+07
